In [0]:
from pyspark.sql.functions import col, expr, trim

# -----------------------------
# 1. Read data
# -----------------------------
df = spark.read.table("mini_project_catalog.`01_bronze`.ns_budget")

# -----------------------------
# 2. Remove duplicates
# -----------------------------
df = df.dropDuplicates()

# -----------------------------
# 3. Replace invalid values ("-") with null
# -----------------------------
df = df.replace("-", None)

# -----------------------------
# 4. Trim all string columns
# -----------------------------
for c in df.columns:
    df = df.withColumn(c, trim(col(c)))

# -----------------------------
# 5. Convert date column
# -----------------------------
if "month" in df.columns:
    df = df.withColumn("month", expr("to_date(month)"))

# -----------------------------
# 6. Convert numeric columns
# -----------------------------
if "budget_amount" in df.columns:
    df = df.withColumn(
        "budget_amount",
        expr("CAST(try_cast(budget_amount AS DOUBLE) AS DOUBLE)")
    )

if "ns_store_id" in df.columns:
    df = df.withColumn("ns_store_id", col("ns_store_id").cast("int"))

# -----------------------------
# 7. Filter valid records
# -----------------------------
df = df.filter(col("ns_store_id").isNotNull())
df = df.filter(col("month").isNotNull())

# -----------------------------
# 8. Standardize text column
# -----------------------------
df = df.withColumn("approved_by", col("approved_by"))

# -----------------------------
# 9. Write to Silver (KEEP ALL COLUMNS)
# -----------------------------
df.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("mini_project_catalog.02_silver.ns_budget")

In [0]:
%sql
SELECT * FROM mini_project_catalog.02_silver.ns_budget LIMIT 10;